# 完整公共品博弈游戏 (Public Goods Game)

## 游戏规则

- **玩家数量**: 16人
- **博弈轮数**: 15轮
- **初始资金**: 1.0
- **收益 b**: 0.1
- **成本 c**: 0.05 × 邻居数
- **网络**: Erdos-Renyi (p=0.3)

## 游戏流程

1. **初始化**: 生成随机网络
2. **每轮循环**:
   - Planner 给出边操作建议
   - 玩家决定接受/拒绝
   - 更新网络
   - Bot 决定合作/背叛
   - 结算收益
   - 计算奖励

In [ ]:
import numpy as np
import random
from typing import Dict, Tuple, List

class Player:
    """玩家类"""
    def __init__(self, player_id, initial_capital=1.0, theta=None):
        self.id = player_id
        self.capital = initial_capital
        self.decision = 0  # 0=背叛, 1=合作
        self.neighbors = []
        self.theta = theta if theta is not None else np.random.randn()
    
    def reset(self, initial_capital=1.0):
        self.capital = initial_capital
        self.decision = 0
    
    def get_properties(self):
        return {
            'id': self.id,
            'capital': self.capital,
            'decision': self.decision,
            'degree': len(self.neighbors),
            'theta': self.theta
        }

## Bot 合作决策 (基于论文参数)

In [ ]:
class BotDecision:
    """Bot 合作决策 (论文参数)"""
    
    # 标准化统计量
    MEAN_XS = 4.55
    STD_XS = 1.78
    MEAN_XN = 2.50
    STD_XN = 1.67
    MEAN_XR = 0.55
    STD_XR = 0.32
    
    # 合作参数 (Round 1)
    BETA0_INIT = 1.807
    BETA1_INIT = 0.818
    
    # 合作参数 (Round > 1)
    BETA0 = -0.010
    BETA1 = -0.75
    BETA2 = 1.16
    BETA3 = 0.46
    
    @staticmethod
    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-x))
    
    @staticmethod
    def decide_cooperation(
        theta: float,
        round_idx: int,
        xs: float,    # 邻居数
        xn: float,    # 合作邻居数
        xr: float,    # 合作率
        capital: float,
        cost: float
    ) -> int:
        """决定是否合作"""
        # 破产保护
        if cost * xs > capital:
            return 0
        
        if round_idx == 1:
            z = BotDecision.BETA0_INIT + BotDecision.BETA1_INIT * theta
        else:
            # 标准化
            xs_std = (xs - BotDecision.MEAN_XS) / BotDecision.STD_XS
            xn_std = (xn - BotDecision.MEAN_XN) / BotDecision.STD_XN
            xr_std = (xr - BotDecision.MEAN_XR) / BotDecision.STD_XR
            
            z = (BotDecision.BETA0 + 
                 BotDecision.BETA1 * xs_std + 
                 BotDecision.BETA2 * xn_std + 
                 BotDecision.BETA3 * xr_std + 
                 theta)
        
        prob = np.clip(BotDecision.sigmoid(z), 0.001, 0.999)
        return 1 if np.random.random() < prob else 0
    
    @staticmethod
    def decide_accept(valence: int, other_decision: int) -> bool:
        """决定是否接受建议"""
        accept_probs = {
            (-1, 0): 0.774,  # 断开 + 对方背叛
            (-1, 1): 0.085,  # 断开 + 对方合作
            (1, 0): 0.287,   # 添加 + 对方背叛
            (1, 1): 0.909,   # 添加 + 对方合作
        }
        prob = accept_probs.get((valence, other_decision), 0.0)
        return np.random.random() < prob

## 完整游戏环境

In [ ]:
class GameEnvironment:
    """完整游戏环境"""
    def __init__(self, n_players=16, n_rounds=15, init_capital=1.0, 
                 benefit=0.1, cost=0.05, erdos_p=0.3, seed=None):
        self.n_players = n_players
        self.n_rounds = n_rounds
        self.init_capital = init_capital
        self.benefit = benefit
        self.cost = cost
        self.erdos_p = erdos_p
        self.seed = seed
        
        self.players = [Player(i, init_capital) for i in range(n_players)]
        self.adj_matrix = None
        self.round_idx = 0
    
    def reset(self):
        """重置游戏"""
        if self.seed is not None:
            np.random.seed(self.seed)
            random.seed(self.seed)
        
        for player in self.players:
            player.reset(self.init_capital)
        
        self._generate_network()
        
        # 第一轮: Bot 决策
        self._bot_decisions()
        self._apply_payoffs()
        
        self.round_idx = 1
        return self._get_state()
    
    def _generate_network(self):
        """生成随机网络"""
        self.adj_matrix = np.zeros((self.n_players, self.n_players))
        
        for i in range(self.n_players):
            for j in range(i+1, self.n_players):
                if np.random.random() < self.erdos_p:
                    self.adj_matrix[i, j] = 1
                    self.adj_matrix[j, i] = 1
        
        for i, player in enumerate(self.players):
            player.neighbors = np.where(self.adj_matrix[i] == 1)[0].tolist()
    
    def _bot_decisions(self):
        """所有 Bot 决定合作/背叛"""
        for i, player in enumerate(self.players):
            degree = len(player.neighbors)
            
            # 计算合作邻居数
            coop_neighbors = sum(1 for n in player.neighbors if self.players[n].decision == 1)
            coop_rate = coop_neighbors / degree if degree > 0 else 0.0
            
            player.decision = BotDecision.decide_cooperation(
                theta=player.theta,
                round_idx=self.round_idx,
                xs=degree,
                xn=coop_neighbors,
                xr=coop_rate,
                capital=player.capital,
                cost=self.cost
            )
    
    def _apply_payoffs(self):
        """结算本轮收益"""
        for i, player in enumerate(self.players):
            if player.decision == 1:  # 合作
                # 付出成本
                cost = self.cost * len(player.neighbors)
                player.capital -= cost
                
                # 邻居获得收益
                for j in player.neighbors:
                    self.players[j].capital += self.benefit
    
    def _get_state(self):
        return {
            'round': self.round_idx,
            'adj_matrix': self.adj_matrix.copy(),
            'players': [p.get_properties() for p in self.players]
        }
    
    def get_player_features(self):
        """获取玩家特征"""
        features = []
        for player in self.players:
            degree = len(player.neighbors)
            coop_neighbors = sum(1 for n in player.neighbors if self.players[n].decision == 1)
            coop_rate = coop_neighbors / degree if degree > 0 else 0.0
            
            features.append({
                'capital': player.capital,
                'decision': player.decision,
                'degree': degree,
                'coop_neighbors': coop_neighbors,
                'coop_rate': coop_rate,
                'theta': player.theta
            })
        return features
    
    def step(self, recommendations: Dict[Tuple, int]):
        """
        执行一步:
        1. Planner 给出建议
        2. 玩家决定接受/拒绝
        3. 更新网络
        4. Bot 决策
        5. 结算收益
        6. 计算奖励
        """
        # 1. 处理建议
        for (i, j), action in recommendations.items():
            if action == 0:
                continue
            
            # 随机选择一个玩家作为决策者
            decision_player = i if np.random.random() < 0.5 else j
            other_player = j if decision_player == i else i
            other_decision = self.players[other_player].decision
            
            # 玩家决定是否接受
            if BotDecision.decide_accept(action, other_decision):
                if action == 1:  # 添加边
                    self.adj_matrix[i, j] = 1
                    self.adj_matrix[j, i] = 1
                    self.players[i].neighbors.append(j)
                    self.players[j].neighbors.append(i)
                elif action == -1:  # 断开边
                    self.adj_matrix[i, j] = 0
                    self.adj_matrix[j, i] = 0
                    if j in self.players[i].neighbors:
                        self.players[i].neighbors.remove(j)
                    if i in self.players[j].neighbors:
                        self.players[j].neighbors.remove(i)
        
        # 2. Bot 决策
        self._bot_decisions()
        
        # 3. 结算收益
        self._apply_payoffs()
        
        # 4. 计算奖励
        avg_capital = np.mean([p.capital for p in self.players])
        
        self.round_idx += 1
        done = self.round_idx >= self.n_rounds
        
        return self._get_state(), avg_capital, done

## Planner

In [ ]:
class BasePlanner:
    def get_recommendations(self, adj_matrix, player_features):
        raise NotImplementedError

class StaticPlanner(BasePlanner):
    def get_recommendations(self, adj_matrix, player_features):
        n = adj_matrix.shape[0]
        return {(i,j):0 for i in range(n) for j in range(i+1,n)}

class RandomPlanner(BasePlanner):
    def __init__(self, change_prob=0.3):
        self.change_prob = change_prob
    
    def get_recommendations(self, adj_matrix, player_features):
        n = adj_matrix.shape[0]
        recs = {}
        for i in range(n):
            for j in range(i+1, n):
                if np.random.random() < self.change_prob:
                    recs[(i,j)] = 1 if adj_matrix[i,j]==0 else -1
                else:
                    recs[(i,j)] = 0
        return recs

class CoopClusteringPlanner(BasePlanner):
    def __init__(self, ct=0.5, dt=0.3):
        self.connect_threshold = ct
        self.disconnect_threshold = dt
    
    def get_recommendations(self, adj_matrix, player_features):
        n = adj_matrix.shape[0]
        coop_rates = []
        for pf in player_features:
            if pf['degree'] > 0:
                coop_rates.append(pf['coop_neighbors'] / pf['degree'])
            else:
                coop_rates.append(0.0)
        
        recs = {}
        for i in range(n):
            for j in range(i+1, n):
                avg = (coop_rates[i] + coop_rates[j]) / 2
                if adj_matrix[i,j] == 0:
                    recs[(i,j)] = 1 if avg > self.connect_threshold else 0
                else:
                    recs[(i,j)] = -1 if avg < self.disconnect_threshold else 0
        return recs

## 完整游戏测试

In [ ]:
# 测试完整游戏
np.random.seed(42)

env = GameEnvironment(n_players=16, n_rounds=15, seed=42)
planner = RandomPlanner(change_prob=0.3)

# 重置游戏
state = env.reset()
print(f"Round 0 (initial): 合作率={np.mean([p['decision'] for p in state['players']]):.2%}")

# 模拟几轮
for round_idx in range(1, 6):
    player_features = env.get_player_features()
    recommendations = planner.get_recommendations(env.adj_matrix, player_features)
    
    state, reward, done = env.step(recommendations)
    coop_rate = np.mean([p['decision'] for p in state['players']])
    avg_capital = np.mean([p['capital'] for p in state['players']])
    
    print(f"Round {round_idx}: 合作率={coop_rate:.2%}, 平均资金={avg_capital:.3f}, 奖励={reward:.3f}")
    
    if done:
        break